# Análisis Pre-Partido — Scouting del Rival
Forma reciente de Chaco, scouting del rival, head-to-head y tabla de posiciones.  
Genera gráficos y el informe narrativo de scouting con Claude.

In [16]:
import os, sys

# Recarga automática de módulos src/ al editarlos (sin reiniciar kernel)
%load_ext autoreload
%autoreload 2

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import config
from src.scraping.rivals_scraper import scrape_rival_recent, rival_already_scraped, scrape_rival_managers
from src.scraping.standings_scraper import fetch_standings, load_standings
from src.analysis.pre_match import (
    chaco_recent_form, chaco_form_summary,
    rival_form, rival_form_summary,
    rival_tactical_profile, rival_key_players, rival_match_keys,
    head_to_head, head_to_head_summary, head_to_head_goalscorers,
    discipline_alert, rival_best_xi, rival_formations_used, rival_discipline_profile,
    rival_manager_timeline, rival_manager_summary,
    rival_home_away_split, rival_suspension_risk,
)
from src.analysis.post_match import load_match_data, xg_analysis, player_performance_table
from src.analysis.season import (
    is_home_match, threat_by_situation, shot_entropy, assists_network,
)
from src.visualization.pre_match import (
    plot_form_comparison, plot_xg_trend, plot_stats_radar, plot_rival_players,
    plot_head_to_head, plot_discipline_alert,
    plot_rival_best_xi, plot_rival_formations, plot_rival_discipline,
    plot_rival_managers, plot_rival_home_away,
)
from src.visualization.season import (
    save_fig, plot_match_stats, plot_shotmap_4grid_match,
    plot_threat_by_situation, plot_shot_entropy, plot_assists_network,
)
from src.visualization.match import plot_player_ratings_match
from src.reports.pre_match import pre_match_report

# ============================================================
# CONFIGURAR ACÁ
SEASON         = '2026'
NEXT_MATCH_KEY = '2026-07-20_vs_san_miguel_h'   # match_key del próximo partido
RIVAL_SLUG     = 'san_miguel'                    # clave en config.RIVALS
CHACO_LAST_N   = 5                             # últimos N partidos de Chaco para forma
GENERATE_REPORT = False
# ============================================================

rival_info = config.RIVALS.get(RIVAL_SLUG, {})
RIVAL_NAME = rival_info.get('name', RIVAL_SLUG.replace('_',' ').title())
_rival_clean = NEXT_MATCH_KEY.split('_vs_',1)[-1] if '_vs_' in NEXT_MATCH_KEY else RIVAL_SLUG
OUTPUTS_DIR = config.pre_outputs_dir(_rival_clean, season=SEASON)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f'Próximo partido : {NEXT_MATCH_KEY}')
print(f'Rival           : {RIVAL_NAME} (slug: {RIVAL_SLUG})')
print(f'Outputs         : {OUTPUTS_DIR}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Próximo partido : 2026-07-20_vs_san_miguel_h
Rival           : San Miguel (slug: san_miguel)
Outputs         : c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\outputs\temporadas\2026\pre_partido\san_miguel_h


## 1. Tabla de posiciones

In [17]:
# Intentar cargar local primero; si no existe, scrapeamos
standings = load_standings(SEASON)
if standings is None:
    print('Descargando tabla...')
    standings = fetch_standings(SEASON, headless=True)
else:
    print('Tabla cargada desde CSV local.')

if standings is not None:
    print(standings[['position','team_name','matches_played','wins','draws','losses','goal_diff','points']].to_string(index=False))

Tabla cargada desde CSV local.
 position                   team_name  matches_played  wins  draws  losses  goal_diff  points
        1     Club Ferro Carril Oeste              19    11      4       4          8      37
        2             Deportivo Morón              19    10      4       5         10      34
        3         Club Atletico Colón              19     8      8       3          8      32
        4           Ciudad de Bolivar              19     7      9       3          5      30
        5                   Los Andes              19     7      8       4          9      29
        6             Almirante Brown              19     8      5       6          2      29
        7            Deportivo Madryn              19     7      7       5          5      28
        8 Estudiantes de Buenos Aires              19     7      6       6         -1      27
        9                  Godoy Cruz              19     6      7       6          2      25
       10                  Sa

## 2. Forma reciente de Chaco

In [18]:
chaco_form  = chaco_recent_form(n=CHACO_LAST_N, season=SEASON)
chaco_summ  = chaco_form_summary(chaco_form)

print(f'Últimos {CHACO_LAST_N} partidos de Chaco:')
print(f"Forma: {chaco_summ.get('forma_str')}")
print(f"GF {chaco_summ.get('goles_favor')} | GC {chaco_summ.get('goles_contra')}")
print(f"xG prom {chaco_summ.get('xg_favor')} | xGA prom {chaco_summ.get('xg_contra')}")
print(chaco_form[['date','opponent','venue','result','chaco_goals','rival_goals','chaco_xg','rival_xg']].to_string(index=False))

Últimos 5 partidos de Chaco:
Forma: LLWDW
GF 6 | GC 6
xG prom 0.9 | xGA prom 1.26
      date           opponent     venue result  chaco_goals  rival_goals  chaco_xg  rival_xg
2026-06-07 Ferro Carril Oeste     Local      L            1            2      0.44      0.90
2026-06-14  Racing De Cordoba Visitante      L            0            2      0.14      3.08
2026-06-20              Colon     Local      W            1            0      0.23      0.33
2026-07-07           Acassuso     Local      D            1            1      1.86      1.22
2026-07-11          San Telmo Visitante      W            3            1      1.85      0.76


## 3. Scrapear partidos del rival (si faltan)

In [19]:
# Baja TODOS los partidos jugados por el rival en la temporada (no solo los
# últimos N) — necesario para que formaciones/mejor once/disciplina se calculen
# sobre la muestra completa, no sobre 5 partidos. Es incremental: los partidos
# ya descargados se saltean sin llamadas de red, así que correr esto de nuevo
# más adelante en la temporada solo baja los partidos nuevos.
scraped = scrape_rival_recent(RIVAL_SLUG, n_matches=None, season=SEASON, headless=False)
print(f'\nPartidos de {RIVAL_NAME} disponibles: {len(scraped)}')

# DT de cada partido (endpoint aparte /managers) para detectar cambios de
# entrenador. Idempotente: los que ya tienen managers.json se saltean.
scrape_rival_managers(RIVAL_SLUG, season=SEASON, headless=False)


Scrapeando TODOS los partidos de San Miguel en la temporada 2026 (id=264945)...
  Encontrados 20 partidos en Primera Nacional.

  [1/20] Ya existe: 2026-02-13_vs_central_norte_de_salta_h
  [2/20] Ya existe: 2026-02-21_vs_club_ferro_carril_oeste_a
  [3/20] Ya existe: 2026-02-28_vs_chaco_for_ever_h
  [4/20] Ya existe: 2026-03-14_vs_san_telmo_h
  [5/20] Ya existe: 2026-03-21_vs_racing_de_cordoba_h
  [6/20] Ya existe: 2026-03-28_vs_colegiales_a
  [7/20] Ya existe: 2026-04-04_vs_club_atletico_colon_a
  [8/20] Ya existe: 2026-04-11_vs_deportivo_moron_h
  [9/20] Ya existe: 2026-04-18_vs_godoy_cruz_a
  [10/20] Ya existe: 2026-04-25_vs_los_andes_h
  [11/20] Ya existe: 2026-04-30_vs_all_boys_a
  [12/20] Ya existe: 2026-05-09_vs_estudiantes_de_buenos_air_h
  [13/20] Ya existe: 2026-05-17_vs_ca_mitre_a
  [14/20] Ya existe: 2026-05-23_vs_almirante_brown_h
  [15/20] Ya existe: 2026-05-31_vs_ciudad_de_bolivar_a
  [16/20] Ya existe: 2026-06-06_vs_defensores_de_belgrano_h
  [17/20] Ya existe: 2026-06-1

['2026-02-13_vs_central_norte_de_salta_h',
 '2026-02-21_vs_club_ferro_carril_oeste_a',
 '2026-02-28_vs_chaco_for_ever_h',
 '2026-03-14_vs_san_telmo_h',
 '2026-03-21_vs_racing_de_cordoba_h',
 '2026-03-28_vs_colegiales_a',
 '2026-04-04_vs_club_atletico_colon_a',
 '2026-04-11_vs_deportivo_moron_h',
 '2026-04-18_vs_godoy_cruz_a',
 '2026-04-25_vs_los_andes_h',
 '2026-04-30_vs_all_boys_a',
 '2026-05-09_vs_estudiantes_de_buenos_air_h',
 '2026-05-17_vs_ca_mitre_a',
 '2026-05-23_vs_almirante_brown_h',
 '2026-05-31_vs_ciudad_de_bolivar_a',
 '2026-06-06_vs_defensores_de_belgrano_h',
 '2026-06-14_vs_deportivo_madryn_a',
 '2026-06-20_vs_acassuso_a',
 '2026-07-05_vs_central_norte_de_salta_a',
 '2026-07-11_vs_club_ferro_carril_oeste_h']

## 4. Análisis del rival

In [20]:
rform       = rival_form(RIVAL_SLUG, SEASON)
rform_summ  = rival_form_summary(rform)
rprofile    = rival_tactical_profile(RIVAL_SLUG, SEASON)
rplayers    = rival_key_players(RIVAL_SLUG, SEASON)

print(f'Forma reciente de {RIVAL_NAME}:')
print(f"Forma: {rform_summ.get('forma_str')}")
print(f"GF {rform_summ.get('goles_favor')} | GC {rform_summ.get('goles_contra')}")
if not rform.empty:
    print(rform[['date','opponent','venue','result','goals_for','goals_against']].to_string(index=False))

print(f'\nJugadores clave de {RIVAL_NAME}:')
if not rplayers.empty:
    print(rplayers.to_string(index=False))

# Rendimiento local vs visitante del rival. Como el partido es en nuestra
# cancha, lo que importa es su forma de VISITANTE.
split = rival_home_away_split(RIVAL_SLUG, SEASON)
if not split.empty:
    print('Local vs Visitante del rival:')
    print(split.to_string(index=False))
    WE_ARE_HOME = NEXT_MATCH_KEY.endswith('_h')
    p = os.path.join(OUTPUTS_DIR, 'local_visitante_rival.png')
    plot_rival_home_away(split, rival_name=RIVAL_NAME, we_are_home=WE_ARE_HOME, save_path=p)
    print('OK local_visitante_rival.png')


Forma reciente de San Miguel:
Forma: WDWDLDLWDLDDLWDDDWDL
GF 17 | GC 21
      date                  opponent     venue result  goals_for  goals_against
2026-02-13    Central Norte De Salta     Local      W          1              0
2026-02-21   Club Ferro Carril Oeste Visitante      D          1              1
2026-02-28            Chaco For Ever     Local      W          2              0
2026-03-14                 San Telmo     Local      D          1              1
2026-03-21         Racing De Cordoba     Local      L          1              2
2026-03-28                Colegiales Visitante      D          0              0
2026-04-04       Club Atletico Colon Visitante      L          0              3
2026-04-11           Deportivo Moron     Local      W          2              1
2026-04-18                Godoy Cruz Visitante      D          0              0
2026-04-25                 Los Andes     Local      L          0              2
2026-04-30                  All Boys Visitante  

## 5. Mejor once y formaciones del rival

In [21]:
# "Mejor once": el arquero + los 10 de campo con mas titularidades en la
# muestra scrapeada (no es la alineacion oficial confirmada del proximo partido).
best_xi = rival_best_xi(RIVAL_SLUG, SEASON)
if not best_xi.empty:
    print(f'Mejor once de {RIVAL_NAME} (segun titularidades recientes):')
    print(best_xi[['player_name','position','starts','matches_played','avg_rating','goals','assists']]
         .to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'mejor_once_rival.png')
    plot_rival_best_xi(best_xi, rival_name=RIVAL_NAME, save_path=p)
    print('OK mejor_once_rival.png')

# Formaciones usadas en sus ultimos partidos scrapeados.
formations = rival_formations_used(RIVAL_SLUG, SEASON)
if not formations.empty:
    print(f'Formaciones de {RIVAL_NAME}:')
    print(formations.to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'formaciones_rival.png')
    plot_rival_formations(formations, rival_name=RIVAL_NAME, save_path=p)
    print('OK formaciones_rival.png')

# Linea de tiempo del DT del rival (cambio de tecnico en la temporada?).
mgr_tl = rival_manager_timeline(RIVAL_SLUG, SEASON)
if not mgr_tl.empty:
    spells = rival_manager_summary(mgr_tl)
    print(f'DT de {RIVAL_NAME} por periodo:')
    for sp in spells:
        print("  " + sp["manager"] + ": " + sp["from_date"] + " a " + sp["to_date"] + " (" + str(sp["matches"]) + " pj)")
    p = os.path.join(OUTPUTS_DIR, 'dt_timeline_rival.png')
    plot_rival_managers(mgr_tl, rival_name=RIVAL_NAME, save_path=p)
    print('OK dt_timeline_rival.png')
else:
    print('Sin datos de DT descargados (correr scrape_rival_managers).')


Mejor once de San Miguel (segun titularidades recientes):
   player_name position  starts  matches_played  avg_rating  goals  assists
J. M. Lungarzo        G      12              12        6.98    0.0      0.0
     K. Ceceri        D      19              20        6.94    1.0      0.0
    F. Cardozo        D      14              19        6.84    0.0      0.0
       A. Cruz        D      13              17        6.78    0.0      1.0
   D. Rentería        D      10              14        6.95    0.0      0.0
    I. Ramírez        M      16              17        6.73    1.0      0.0
    J. Ferrero        M      14              20        6.76    2.0      2.0
       M. Oses        M      12              18        6.99    0.0      1.0
      M. Serra        M      11              20        6.59    0.0      0.0
   B. I. Nasta        F      15              16        6.92    5.0      2.0
    L. Delgado        F      13              18        6.76    3.0      0.0
OK mejor_once_rival.png
Formac

## 6. Historial cara a cara (H2H)
Todos los enfrentamientos con datos disponibles (no solo esta temporada) —
matchea por `team_id` de SofaScore, no por el slug del match_key (que a veces
cambia de una temporada a otra).

In [22]:
h2h = head_to_head(RIVAL_SLUG)
h2h_summ = head_to_head_summary(h2h)
chaco_scorers, rival_scorers = head_to_head_goalscorers(h2h)

if not h2h.empty:
    print(f"Historial vs {RIVAL_NAME}: {h2h_summ['partidos']} partidos "
         f"({h2h_summ['victorias']}V-{h2h_summ['empates']}E-{h2h_summ['derrotas']}D) "
         f"| GF {h2h_summ['goles_favor']} - GC {h2h_summ['goles_contra']}")
    print(h2h[['date','season','venue','chaco_goals','rival_goals','result']].to_string(index=False))

    if not chaco_scorers.empty:
        print(f'\nGoleadores de Chaco vs {RIVAL_NAME}:')
        print(chaco_scorers.to_string(index=False))
    if not rival_scorers.empty:
        print(f'\nGoleadores de {RIVAL_NAME} vs Chaco:')
        print(rival_scorers.to_string(index=False))

    _mgr_note = None
    _tl = rival_manager_timeline(RIVAL_SLUG, SEASON)
    if not _tl.empty:
        _sp = rival_manager_summary(_tl)
        _nch = int(_tl['changed'].sum())
        _mgr_note = f"DT actual del rival: {_sp[-1]['manager']}" + (f' ({_nch} cambios en {SEASON})' if _nch else '')
    p = os.path.join(OUTPUTS_DIR, 'cara_a_cara.png')
    plot_head_to_head(h2h, chaco_scorers, rival_scorers, rival_name=RIVAL_NAME, save_path=p,
                      summary=h2h_summ, manager_note=_mgr_note)
    print('✅ cara_a_cara.png')
else:
    print(f'Sin historial previo con datos vs {RIVAL_NAME} (primer cruce registrado).')

Historial vs San Miguel: 1 partidos (0V-0E-1D) | GF 0 - GC 2
      date season     venue  chaco_goals  rival_goals result
2026-02-28   2026 Visitante            0            2      L

Goleadores de San Miguel vs Chaco:
    player  goals
J. Ferrero      1
L. Delgado      1
✅ cara_a_cara.png


### Detalle del último cruce
Reutiliza el análisis de post-partido (`src/analysis/post_match.py` y sus
gráficos) sobre el `match_key` del cruce más reciente: comparación de
estadísticas, shotmap del partido y ratings — no solo el resultado.

In [23]:
if not h2h.empty:
    last_row = h2h.iloc[-1]
    last_mk, last_season = last_row['match_key'], last_row['season']
    print(f"Último cruce: {last_mk}  ({last_row['venue']}, resultado {last_row['result']})")

    data = load_match_data(last_mk, season=last_season)
    xg = xg_analysis(last_mk, data=data, season=last_season)
    if xg:
        print(xg['verdict'])
        print(f"xG: Chaco {xg['chaco_xg']} - {xg['rival_xg']} {RIVAL_NAME}")

    home_match = is_home_match(last_mk)

    if data.get('stats') is not None:
        home_label = 'Chaco For Ever' if home_match else RIVAL_NAME
        away_label = RIVAL_NAME if home_match else 'Chaco For Ever'
        p = os.path.join(OUTPUTS_DIR, 'ultimo_cruce_stats.png')
        plot_match_stats(data['stats'], home_label=home_label, away_label=away_label,
                         is_chaco_home=home_match, save_path=p)
        print('✅ ultimo_cruce_stats.png')

    if data.get('shotmap') is not None:
        p = os.path.join(OUTPUTS_DIR, 'ultimo_cruce_shotmap.png')
        plot_shotmap_4grid_match(data['shotmap'], is_home=home_match, chaco_label='Chaco For Ever',
                                 rival_label=RIVAL_NAME, match_label=last_mk, save_path=p)
        print('✅ ultimo_cruce_shotmap.png')

    perf = player_performance_table(last_mk, data=data, season=last_season)
    if not perf.empty:
        p = os.path.join(OUTPUTS_DIR, 'ultimo_cruce_ratings.png')
        plot_player_ratings_match(perf, chaco_label='Chaco For Ever', save_path=p)
        print('✅ ultimo_cruce_ratings.png')
        top = perf.dropna(subset=['rating']).sort_values('rating', ascending=False).head(3)
        print('\nMejores de Chaco en ese partido:')
        print(top[['player_name','rating','goals','assists']].to_string(index=False))

    # Goleadores del rival en ESE partido puntual (no todo el historial acumulado)
    if data.get('incidents') is not None:
        inc = data['incidents']
        rival_goals_last = inc[(inc['type'] == 'goal') & (inc['is_home'] != home_match)]
        if not rival_goals_last.empty:
            nombres = rival_goals_last['player'].dropna().astype(str).tolist()
            print(f"\nGoleadores de {RIVAL_NAME} en ese partido: " + ", ".join(nombres))

Último cruce: 2026-02-28_vs_san_miguel_a  (Visitante, resultado L)
Chaco ganó en xG (1.39 vs 0.91) — derrota injusta según xG
xG: Chaco 1.39 - 0.91 San Miguel
✅ ultimo_cruce_stats.png
✅ ultimo_cruce_shotmap.png
✅ ultimo_cruce_ratings.png

Mejores de Chaco en ese partido:
 player_name  rating  goals  assists
  M. Silvera     7.5    0.0      0.0
G. Melgarejo     7.1    0.0      0.0
   B. Nievas     7.0    0.0      0.0

Goleadores de San Miguel en ese partido: J. Ferrero, L. Delgado


## 7. Perfil ofensivo del rival
Reutiliza tal cual las funciones de temporada (`threat_by_situation`,
`shot_entropy`, `assists_network` de `src/analysis/season.py`) pero aplicadas
a los partidos DEL RIVAL (ya con la temporada completa scrapeada): por dónde
suele atacar más y quiénes son sus goleadores/asistidores.

In [24]:
rival_mks = rival_match_keys(RIVAL_SLUG, SEASON)
rival_base = os.path.join(config.rivals_dir(SEASON), RIVAL_SLUG)

rival_threat = threat_by_situation(rival_mks, rival_base)
if not rival_threat.empty:
    print(f'Amenaza por tipo de jugada de {RIVAL_NAME}:')
    print(rival_threat.to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'amenaza_rival.png')
    plot_threat_by_situation(rival_threat, season=SEASON,
                             title=f'Amenaza por Tipo de Jugada – {RIVAL_NAME}', save_path=p)
    print('✅ amenaza_rival.png')

rival_entropy = shot_entropy(rival_mks, rival_base)
if rival_entropy['n_shots']:
    print(f"\nEntropía de tiros de {RIVAL_NAME}: {rival_entropy['entropy']} / "
         f"{rival_entropy['max_entropy']} bits ({rival_entropy['pct_of_max']}% del máximo)")
    p = os.path.join(OUTPUTS_DIR, 'entropia_rival.png')
    plot_shot_entropy(rival_entropy, season=SEASON, team_label=RIVAL_NAME, save_path=p)
    print('✅ entropia_rival.png')

rival_an, rival_ae = assists_network(rival_mks, rival_base)
if not rival_an.empty:
    print(f'\nGoleadores/asistidores de {RIVAL_NAME} en la temporada:')
    print(rival_an.to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'red_asistencias_rival.png')
    plot_assists_network(rival_an, rival_ae, season=SEASON,
                         title=f'Red de Asistencias – {RIVAL_NAME}', save_path=p)
    print('✅ red_asistencias_rival.png')

Amenaza por tipo de jugada de San Miguel:
    situation  shots_for  shots_against  goals_for  goals_against  xg_for  xg_against
Juego abierto        106            120         10             14    8.40        9.98
       Córner         22             22          2              2    2.98        2.74
   Tiro libre          9             10          0              0    0.65        0.33
      Lateral         39             35          2              2    1.56        3.61
Pelota parada         15             10          0              1    1.26        1.59
        Penal          2              2          2              2    1.52        1.52
✅ amenaza_rival.png

Entropía de tiros de San Miguel: 1.739 / 4.644 bits (37.4% del máximo)
✅ entropia_rival.png

Goleadores/asistidores de San Miguel en la temporada:
       player  goals  assists
  B. I. Nasta      5        2
   L. Delgado      3        0
   J. Ferrero      2        2
    D. Juárez      2        0
   F. Almeida      2        0
R. Etche

## 7b. Cadena de Markov del rival (cómo gestiona el marcador)
Reutiliza `markov_game_state` de la temporada (la misma cadena que usamos para
Chaco) aplicada a los partidos del rival: qué le pasa cuando empata, cuando se
pone en ventaja de un gol y cuando va perdiendo por uno. Genera `markov_rival.png`
+ las lecturas de scouting en texto.

In [25]:
# Cadena de Markov del estado del marcador DEL RIVAL — el mismo modelo que
# usamos para Chaco en la temporada (src/analysis/season.markov_game_state),
# reutilizado sobre los partidos del rival. Lee cómo se mueve el marcador en
# sus partidos: si desde el empate suele golpear primero o recibir, si sostiene
# las ventajas de un gol o se las empatan, y si reacciona o se hunde cuando pierde.
from src.analysis.pre_match import rival_markov, rival_markov_summary
from src.visualization.season import plot_markov_transitions

rmat, rtrans = rival_markov(RIVAL_SLUG, SEASON)
if rmat is not None and not rmat.empty:
    rmk_summ = rival_markov_summary(rmat, rtrans)
    print(f'Cadena de Markov de {RIVAL_NAME} (estado del marcador, perspectiva propia):')
    print(rmat.round(2).to_string())
    print('\nLecturas de scouting:')
    for _k in ('desde_empate', 'ganando_por_1', 'perdiendo_por_1'):
        _d = rmk_summ.get(_k)
        if _d:
            print(f"  [{_k}] (n={_d.get('n')}): {_d['lectura']}")
    p = os.path.join(OUTPUTS_DIR, 'markov_rival.png')
    plot_markov_transitions(rmat, rtrans, season=SEASON,
                            title=f'Cadena de Markov del Marcador - {RIVAL_NAME}', save_path=p)
    print('\n✅ markov_rival.png')
else:
    print(f'Sin partidos scrapeados de {RIVAL_NAME} todavía — corré primero la celda de scraping (sección 3).')

Cadena de Markov de San Miguel (estado del marcador, perspectiva propia):
      S--    S-    S0    S+   S++
S--  0.75  0.25  0.00  0.00  0.00
S-   0.67  0.00  0.33  0.00  0.00
S0   0.00  0.37  0.00  0.63  0.00
S+   0.00  0.00  0.78  0.00  0.22
S++  0.00  0.00  0.00  0.00  0.00

Lecturas de scouting:
  [desde_empate] (n=19): Desde el 0-0 suele golpear primero: se pone en ventaja el 63% de las veces.
  [ganando_por_1] (n=9): Cuando se pone 1-0 le cuesta cerrarlo: lo alcanzan el 78% de las veces.
  [perdiendo_por_1] (n=6): Si se pone 0-1 se cae: el 67% de las veces recibe el segundo antes de reaccionar.

✅ markov_rival.png


## 7c. Comparación Chaco vs rival (equipo y jugadores)
Perfil de juego de los dos equipos, promedios por partido: **radar ampliado**,
**comparación ofensiva** y **defensiva**, y el desglose de **remates realizados**
y **remates recibidos**. Cierra con un **radar de jugadores cruzado** (un jugador
de Chaco vs una figura del rival, escalado al percentil de la liga). Todo se guarda
en la carpeta del partido.

In [ ]:
# Comparación de EQUIPOS Chaco vs rival (perfil de juego, promedios por partido)
# + radar ampliado + paneles ofensivo/defensivo/remates + radar de jugadores cruzado.
from src.analysis.pre_match import (
    chaco_two_way_profile, rival_two_way_profile, cross_team_player_pool,
)
from src.visualization.pre_match import (
    plot_team_radar_extended, plot_offensive_comparison,
    plot_defensive_comparison, plot_shots_comparison,
)
from src.analysis.players import compare_players
from src.visualization.players import plot_players_radar

# --- Dupla del radar de jugadores (editá si querés otra; deben estar en el pool) ---
JUGADOR_CHACO = None   # ej. 'D. Valdez'   (None = elige el de más goles/90 de Chaco)
JUGADOR_RIVAL = None   # ej. 'S. Bellatti' (None = elige el de más goles/90 del rival)

cprof = chaco_two_way_profile(SEASON)
rprof = rival_two_way_profile(RIVAL_SLUG, SEASON)

if rprof.get('n_matches'):
    plot_team_radar_extended(cprof, rprof, RIVAL_NAME,
                             save_path=os.path.join(OUTPUTS_DIR, 'radar_ampliado.png'))
    plot_offensive_comparison(cprof, rprof, RIVAL_NAME,
                              save_path=os.path.join(OUTPUTS_DIR, 'comparacion_ofensiva.png'))
    plot_defensive_comparison(cprof, rprof, RIVAL_NAME,
                              save_path=os.path.join(OUTPUTS_DIR, 'comparacion_defensiva.png'))
    plot_shots_comparison(cprof, rprof, RIVAL_NAME, side='for',
                          save_path=os.path.join(OUTPUTS_DIR, 'remates_realizados.png'))
    plot_shots_comparison(cprof, rprof, RIVAL_NAME, side='against',
                          save_path=os.path.join(OUTPUTS_DIR, 'remates_recibidos.png'))
    print('✅ radar_ampliado, comparacion_ofensiva, comparacion_defensiva, remates_realizados, remates_recibidos')

    # Radar de jugadores cruzado (uno de Chaco vs uno del rival, escala liga)
    pool = cross_team_player_pool(RIVAL_SLUG, SEASON)
    rival_mask = pool['team_name'].str.contains(RIVAL_NAME, case=False, na=False)
    # Jugadores de Chaco disponibles: saca a los que se fueron (config.JUGADORES_FUERA)
    chaco_disp = config.excluir_jugadores_fuera(pool[pool['is_chaco']])
    if JUGADOR_CHACO is None:
        _c = chaco_disp.sort_values('goals_p90', ascending=False)
        JUGADOR_CHACO = _c.iloc[0]['player_name'] if not _c.empty else None
    if JUGADOR_RIVAL is None:
        _r = pool[rival_mask].sort_values('goals_p90', ascending=False)
        JUGADOR_RIVAL = _r.iloc[0]['player_name'] if not _r.empty else None
    print(f'Radar jugadores: {JUGADOR_CHACO} (Chaco) vs {JUGADOR_RIVAL} ({RIVAL_NAME})')
    cmp = compare_players(JUGADOR_CHACO, JUGADOR_RIVAL, pool_df=pool)
    if cmp:
        plot_players_radar(cmp, save_path=os.path.join(OUTPUTS_DIR, 'radar_jugadores_cruzado.png'))
        print('✅ radar_jugadores_cruzado.png')
    else:
        print('No encontré alguno de los dos jugadores en el pool (revisá los nombres abajo).')
    print('\nJugadores disponibles para la dupla (por minutos):')
    print('  Chaco :', ', '.join(chaco_disp.sort_values('minutes', ascending=False)['player_name'].head(12)))
    print(f'  {RIVAL_NAME}:', ', '.join(pool[rival_mask].sort_values('minutes', ascending=False)['player_name'].head(12)))
else:
    print(f'Sin partidos scrapeados de {RIVAL_NAME} todavía — corré primero la celda de scraping (sección 3).')

✅ radar_ampliado, comparacion_ofensiva, comparacion_defensiva, remates_realizados, remates_recibidos
Radar jugadores: L. Fernandez (Chaco) vs D. Adín (San Miguel)
✅ radar_jugadores_cruzado.png

Jugadores disponibles para la dupla (por minutos):
  Chaco : D. Valdez, A. Luque, M. Silvera, B. Nievas, G. Pacheco, N. Caprio, I. Enriquez, J. Carrizo, L. Marinucci, A. Ojeda, R. Garay, L. Pruzzo
  San Miguel: K. Ceceri, I. Ramírez, F. Cardozo, L. Delgado, B. I. Nasta, J. Ferrero, M. Oses, J. M. Lungarzo, A. Cruz, M. Serra, D. Rentería, D. Adín


## 8. Disciplina — alerta propia y perfil del rival
Jugadores de Chaco a 1 amarilla de la suspensión (cuenta solo los partidos
jugados ANTES de `NEXT_MATCH_KEY`) + tendencia disciplinaria reciente del
rival (NO es una alerta de suspensión de ellos — no tenemos su acumulado
real de temporada, solo los últimos partidos scrapeados).

In [27]:
alert_df = discipline_alert(NEXT_MATCH_KEY, season=SEASON)
# No alertar por jugadores que ya se fueron (config.JUGADORES_FUERA)
alert_df = config.excluir_jugadores_fuera(alert_df, 'player')

if not alert_df.empty:
    print(f'⚠️  Jugadores en riesgo de suspensión para {NEXT_MATCH_KEY}:')
    print(alert_df.to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'alerta_amonestados.png')
    plot_discipline_alert(alert_df, rival_name=RIVAL_NAME, save_path=p)
    print('✅ alerta_amonestados.png')
else:
    print('Sin jugadores en riesgo de suspensión para este partido.')

# Perfil disciplinario del rival. Como bajamos TODA su temporada, el acumulado
# es real: se puede marcar quien esta a una amarilla de la suspension.
rival_team_disc, rival_player_disc = rival_discipline_profile(RIVAL_SLUG, SEASON)
rival_susp = rival_suspension_risk(RIVAL_SLUG, SEASON)
if not rival_susp.empty:
    print('Rival a 1 amarilla de la suspension:')
    print(rival_susp.to_string(index=False))
if not rival_team_disc.empty:
    print(f'\nPerfil disciplinario reciente de {RIVAL_NAME}:')
    print(rival_team_disc.to_string(index=False))
    if not rival_player_disc.empty:
        print(f'\nMás amonestados de {RIVAL_NAME}:')
        print(rival_player_disc.head(6).to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'disciplina_rival.png')
    plot_rival_discipline(rival_team_disc, rival_player_disc, rival_name=RIVAL_NAME, save_path=p,
                          suspension_df=rival_susp, full_season=True)
    print('✅ disciplina_rival.png')

⚠️  Jugadores en riesgo de suspensión para 2026-07-20_vs_san_miguel_h:
      player  amarillas  rojas  partidos
    A. Luque        4.0    0.0        20
 I. Enriquez        4.0    0.0        19
L. Marinucci        4.0    0.0        15
   B. Guerra        4.0    0.0        10
✅ alerta_amonestados.png
Rival a 1 amarilla de la suspension:
player_name position  yellow_cards  to_ban
    D. Adín        D             4       1
 I. Ramírez        M             4       1
 J. Ferrero        M             4       1

Perfil disciplinario reciente de San Miguel:
                                match_key       date                  opponent  fouls  yellow_cards  red_cards
   2026-02-13_vs_central_norte_de_salta_h 2026-02-13    Central Norte De Salta     13             5          0
  2026-02-21_vs_club_ferro_carril_oeste_a 2026-02-21   Club Ferro Carril Oeste     20             6          0
           2026-02-28_vs_chaco_for_ever_h 2026-02-28            Chaco For Ever     11             2          0


## 9. Visualizaciones

In [28]:
import matplotlib
matplotlib.use('Agg')

if not chaco_form.empty and not rform.empty:
    fig = plot_form_comparison(chaco_form, rform, rival_name=RIVAL_NAME)
    save_fig(fig, os.path.join(OUTPUTS_DIR, 'forma_comparada.png'))
    print('✅ forma_comparada.png')

    fig = plot_xg_trend(chaco_form, rform, rival_name=RIVAL_NAME)
    save_fig(fig, os.path.join(OUTPUTS_DIR, 'tendencia_xg.png'))
    print('✅ tendencia_xg.png')

if rprofile:
    # Calcular perfil promedio de Chaco desde sus últimos partidos
    from src.analysis.season import KEY_METRICS, build_season_table, load_matches
    import pandas as pd
    base_dir = config.season_data_dir(SEASON)
    chaco_profile = {}
    if not chaco_form.empty:
        for mk in chaco_form['match_key']:
            sp = os.path.join(base_dir, mk, 'statistics_clean.csv')
            if os.path.exists(sp):
                df_s = pd.read_csv(sp)
                from src.analysis.season import is_home_match
                col = 'home_val' if is_home_match(mk) else 'away_val'
                for _, r in df_s.iterrows():
                    m = r.get('metric','')
                    try:
                        v = float(r.get(col, 0) or 0)
                        chaco_profile[m] = chaco_profile.get(m, []) + [v]
                    except (ValueError, TypeError):
                        pass
        chaco_profile = {k: round(sum(v)/len(v),2) for k,v in chaco_profile.items() if v}

    if chaco_profile and rprofile:
        fig = plot_stats_radar(chaco_profile, rprofile, rival_name=RIVAL_NAME)
        save_fig(fig, os.path.join(OUTPUTS_DIR, 'radar_stats.png'))
        print('✅ radar_stats.png')

if not rplayers.empty:
    fig = plot_rival_players(rplayers, rival_name=RIVAL_NAME)
    save_fig(fig, os.path.join(OUTPUTS_DIR, 'jugadores_rival.png'))
    print('✅ jugadores_rival.png')

✅ forma_comparada.png
✅ tendencia_xg.png
✅ radar_stats.png
✅ jugadores_rival.png


## 10. Generar informe de scouting (Claude)

In [29]:
if GENERATE_REPORT:
    report = pre_match_report(
        next_match_key=NEXT_MATCH_KEY,
        rival_slug=RIVAL_SLUG,
        chaco_form_summary=chaco_summ,
        rival_form_summary=rform_summ,
        rival_profile=rprofile,
        rival_players_df=rplayers if not rplayers.empty else None,
        h2h_df=h2h if not h2h.empty else None,
        standings_df=standings,
        chaco_scorers_df=chaco_scorers if not chaco_scorers.empty else None,
        rival_scorers_df=rival_scorers if not rival_scorers.empty else None,
        discipline_alert_df=alert_df if not alert_df.empty else None,
    )
    print('\n' + report[:600] + '...')
else:
    print('GENERATE_REPORT=False — informe omitido.')

GENERATE_REPORT=False — informe omitido.
